# Optimization Difficulty Score (tau) — Exploratory Analysis

This notebook demonstrates the behavior of the `OptimizationDifficultyTracker` (`src/models/tau.py`) on synthetic data, reproducing the qualitative shape of **Figure 4 (Tau Metric Justification)** and **Figure 19 (Appendix D: Tau Score Distribution)** from the paper.

It is meant as a sanity-check / teaching tool, not a substitute for the full training run.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import torch
import matplotlib.pyplot as plt

from models.tau import OptimizationDifficultyTracker, TauConfig

## 1. Simulate a 'head' sample (stable, low gradient) vs. a 'tail' sample (unstable, high gradient)

This reproduces the qualitative pattern in paper Figure 19(a): tail categories (e.g. Chimney, Dam) maintain persistently higher tau than head categories (e.g. Airplane, Ship) throughout training.

In [ ]:
np.random.seed(0)
n_iters = 100

tracker = OptimizationDifficultyTracker(TauConfig(ema_decay=0.9))

head_tau_history, tail_tau_history = [], []

for t in range(n_iters):
    # Head sample: low, stable gradient + stable loss
    head_grad = 0.3 + np.random.normal(0, 0.05)
    head_loss = 0.2 + np.random.normal(0, 0.02)

    # Tail sample: high, fluctuating gradient + fluctuating loss (simulates
    # small, background-cluttered, densely-distributed tail-object learning)
    tail_grad = 0.9 + 0.4 * np.sin(t * 0.3) + np.random.normal(0, 0.15)
    tail_loss = 0.7 + 0.3 * np.sin(t * 0.4 + 1) + np.random.normal(0, 0.1)

    tau_batch = tracker.update(
        ['head_sample', 'tail_sample'],
        torch.tensor([max(head_grad, 0.01), max(tail_grad, 0.01)]),
        torch.tensor([max(head_loss, 0.0), max(tail_loss, 0.0)]),
    )
    head_tau_history.append(tau_batch[0].item())
    tail_tau_history.append(tau_batch[1].item())

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(tail_tau_history, color='crimson', label='Tail sample (e.g. Chimney, Dam)')
plt.plot(head_tau_history, color='steelblue', label='Head sample (e.g. Airplane, Ship)')
plt.axhline(np.mean(head_tau_history + tail_tau_history), color='gray', linestyle='--', label='E[tau] threshold')
plt.xlabel('Training Iteration')
plt.ylabel('tau score')
plt.title('Optimization Difficulty Score Evolution: Head vs. Tail')
plt.legend()
plt.tight_layout()
plt.savefig('../figures/tau_evolution_demo.png', dpi=150)
plt.show()

## 2. EMA noise suppression demo

Reproduces paper Figure 4(b): a single transient gradient spike should be smoothed out by the EMA, while genuinely sustained difficulty is preserved.

In [ ]:
tracker2 = OptimizationDifficultyTracker(TauConfig(ema_decay=0.9))
raw_grads, smoothed_tau = [], []

for t in range(100):
    g = 0.3 + np.random.normal(0, 0.03)
    if t == 35:
        g = 2.0  # single transient noise spike
    raw_grads.append(g)
    tau = tracker2.update(['x'], torch.tensor([max(g, 0.01)]), torch.tensor([0.3]))
    smoothed_tau.append(tau[0].item())

plt.figure(figsize=(8, 4))
plt.plot(raw_grads, ':', color='steelblue', label='Raw gradient magnitude')
plt.plot(smoothed_tau, color='crimson', label='EMA-smoothed tau')
plt.axvspan(33, 40, color='orange', alpha=0.2, label='Noise spike (decays within ~10 iters)')
plt.xlabel('Training Iteration')
plt.ylabel('Value')
plt.title('EMA Noise Filtering: Transient Spike vs. Smoothed Signal')
plt.legend()
plt.tight_layout()
plt.show()

## Next steps

- Run this analysis on real per-sample statistics collected from an actual training run (persist `tau_tracker.get_all_tau()` to disk each epoch and load it here).
- Compare against `docs/methodology.md` for the exact equation-to-code mapping.